<a href="https://colab.research.google.com/github/andersonmoraix/anotacoeseconometria/blob/main/aula6_heterocedasticidade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econometria Aplicada [Módulo em Python]

### Heterocedasticidade

## Introdução

Considere o modelo descrito por

$$y = X \beta + \varepsilon \nonumber$$

Como vimos, temos como hipóteses do modelo de regressão linear que

$$ E \left \{ \varepsilon | X  \right \} = E \left \{ \varepsilon  \right \} = 0 \\ V \left \{ \varepsilon | X  \right \} = V \left \{ \varepsilon  \right \} = \sigma^2 I $$

Que dizem que a distribuição condicional dos termos de erro dada a matriz de variáveis explanatórias possui média zero, variância constante e covariância nula. Em particular, isso significa que cada erro possui a mesma variância e que dois erros diferentes não são correlacionados. Essas condições implicam que $E\left \{ \varepsilon_i | x_i  \right \} = 0$, de modo que o modelo corresponde à esperança condicional de $y_i$ dado $x_i$.

A heterocedasticidade ocorre quando a segunda equação não é mais válida. A heterocedasticidade surge se diferentes termos de erro não possuem a mesma variância, assim os elementos da diagonal da matriz de covariância não serão os mesmos. Por exemplo, é possível que diferentes grupos na amostra possuam diferentes variâncias.

## Teste de heterocedasticidade

O Teste Breush-Pagan (BP) para heterocedasticidade é fácil de ser implementado com rotinas de MQO básicas. Após estimar nosso modelo, nós tomamos os resíduos $e_i$. Feito isso, nós regredimos o valor dos resíduos ao quadrado em relação às variáveis explicativas da equação original. Assim, basta verificar o teste $F$ de significância ou utilizar um teste $LM$ multiplicando o $R^2$ da segunda regressão pelo número de observações. A hipótese nula do teste é de homocedasticidade.

Para fazer esse teste no python, basta utilizarmos a função *het_breuschpagan*. Utilizaremos o dataset "hprice2", do pacote do wooldridge, que traz características de imóveis.  

In [ ]:
import wooldridge as woo
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy as pt

hprice = woo.dataWoo('hprice2')

# Modelo:
reg = smf.ols(formula='price ~ crime+rooms+dist', data=hprice)
results = reg.fit()

print(results.summary())

# Teste BP:
y, X = pt.dmatrices('price ~ crime+rooms+dist',
                    data=hprice, return_type='dataframe')

result_bp_lm = sm.stats.diagnostic.het_breuschpagan(results.resid, X)
bp_lm_statistic = result_bp_lm[0]
bp_lm_pval = result_bp_lm[1]
print(f'bp_lm_estatistica: {bp_lm_statistic}\n')
print(f'bp_lm_pvalor: {bp_lm_pval}\n')

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.543
Model:                            OLS   Adj. R-squared:                  0.540
Method:                 Least Squares   F-statistic:                     198.9
Date:                Thu, 15 Jul 2021   Prob (F-statistic):           5.44e-85
Time:                        05:14:29   Log-Likelihood:                -5138.0
No. Observations:                 506   AIC:                         1.028e+04
Df Residuals:                     502   BIC:                         1.030e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -2.958e+04   2601.679    -11.371      0.0

Como o P-valor é bem baixo, podemos rejeitar a hipótese nula de homocedasticidade. Ou seja, há heterocedasticidade nesta estimação.

## Inferência robusta na presença de heterocedasticidade

Na presença de heterocedasticidade, os testes de hipóteses ficam comprometidos. Uma forma de contornar esse problema é corrigindo o cálculo da matriz de covariância. Para fazer isso no python, é preciso alterar o argumento *cov_type* na função **fit**.  Temos as seguintes opções:

    1)'nonrobust' - padrão, sem correção.
    2)'HC0' - Versão clássica da matriz de variância-covariância de White.
    3)'HC1' - Versão robusta da matriz de White.
    4)'HC2' - Versão com correção para pequenas amostras.
    5)'HC3' - Versão refinada da matriz de White.
    
Se a heterocedasticidade realmente for um problema, o teste F sofrerá grandes alterações com a inclusão dessas versões da matriz de White.

In [ ]:
hprice = woo.dataWoo('hprice2')

# Modelo:
reg = smf.ols(formula='price ~ crime+rooms+dist', data=hprice)
results = reg.fit(cov_type = "HC1")

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.543
Model:                            OLS   Adj. R-squared:                  0.540
Method:                 Least Squares   F-statistic:                     87.89
Date:                Thu, 15 Jul 2021   Prob (F-statistic):           1.01e-45
Time:                        05:14:29   Log-Likelihood:                -5138.0
No. Observations:                 506   AIC:                         1.028e+04
Df Residuals:                     502   BIC:                         1.030e+04
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -2.958e+04   4292.051     -6.893      0.0

Comparando com a versão sem nenhuma correção, é possível perceber que o modelo com matriz de White não altera os coeficientes estimados. Entretanto, altera significativamente os erros padrão calculados. A heterocedasticidade é de fato um problema, dada a grande variação do teste F.

## Mínimos quadrados ponderados (Weighted Least Squares)

O estimador de mínimos quadrados ponderados tenta prover uma solução mais eficiente em relação ao MQO (OLS). Ao invés de minimizarmos a soma dos quadrados dos resíduos, nós minimizamos a soma ponderada. Se os pesos forem inversamente proporcionais à variância, o estimador é eficiente.

Para estimar esse modelo no Python, podemos utilizar a função **wls**, do *statsmodels*. Os pesos são definidos no parâmetro *weights*. Por exemplo, mantendo o a estimação que fizemos anteriormente, se assumirmos que a variância é proporcional a variável de distância ("dist"), então o peso ótimo é $1/dist$.

In [ ]:
hprice = woo.dataWoo('hprice2')

# Modelo:
reg = smf.wls(formula='price ~ crime+rooms+dist',
              data=hprice,
              weights = 1/hprice["dist"])
results = reg.fit()

print(results.summary())

                            WLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.420
Model:                            WLS   Adj. R-squared:                  0.417
Method:                 Least Squares   F-statistic:                     121.3
Date:                Thu, 15 Jul 2021   Prob (F-statistic):           4.25e-59
Time:                        05:14:29   Log-Likelihood:                -5279.3
No. Observations:                 506   AIC:                         1.057e+04
Df Residuals:                     502   BIC:                         1.058e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -2.043e+04   2899.578     -7.046      0.0

Entretanto, essa hipótese de que a variância é proporcional a um dos regressores específicamente, é difícil de ser justificada. Normalmente, não sabemos a função de variância.